## PACOTES 

In [1]:
import os
import time

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import ks_2samp

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    f1_score,
    matthews_corrcoef,
    log_loss,
    confusion_matrix
)

from openTSNE import TSNE

## CONF GERAIS

In [ ]:
# =========================================================
# CONFIG
# =========================================================
RANK = 1

TARGET_COL = "status_fraude"

FRAC_NAO_FRAUDE = 0.25

THRESHOLD = 0.50

estado_randomico = 42

NOME_HTML = f"3d_rank_{RANK}_tsne.html"

# HIPERPARÂMETROS GMM
numero_de_componentes = 2
inicializacoes_gausianas = 3
tipo_matriz_covariancia = "full"
erro_numerico = 1e-6

# HIPERPARÂMETROS t-SNE
TSNE_PERPLEXITY = 30
TSNE_N_ITER = 1000
TSNE_EARLY_EXAGGERATION_ITER = 100
TSNE_EARLY_EXAGGERATION = 12
TSNE_EXAGGERATION = 1
TSNE_LEARNING_RATE = "auto"
TSNE_METRIC = "euclidean"
TSNE_INITIALIZATION = "pca"
TSNE_NEGATIVE_GRADIENT_METHOD = "bh"
TSNE_N_JOBS = 7
TSNE_RANDOM_STATE = estado_randomico
TSNE_VERBOSE = True


## 1 COLOCADO

### T-SNE

In [3]:
RANK = 1
NOME_HTML = f"3d_rank_{RANK}_tsne.html"
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# LOAD RANKING 3x3
df_scores = pd.read_csv("3x3_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]
feature_3 = row["Feature_3"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features originais: {feature_1} vs {feature_2} vs {feature_3}")

# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# DATASET BASE
df_base = df[
    [feature_1, feature_2, feature_3, TARGET_COL]
].dropna()


# FRAUDES: 100%
fraudes = df_base[
    df_base[TARGET_COL] == 1
]

# NÃO FRAUDES: FRAÇÃO CONFIGURÁVEL
nao_fraudes = df_base[
    df_base[TARGET_COL] == 0
].sample(
    frac=FRAC_NAO_FRAUDE,
    random_state=estado_randomico
)

# DATASET FINAL
df_model = pd.concat([
    fraudes,
    nao_fraudes
])

df_model = df_model.sample(
    frac=1,
    random_state=estado_randomico
).reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# FEATURES ORIGINAIS
X_original = df_model[
    [feature_1, feature_2, feature_3]
]

y = df_model[TARGET_COL]

# SCALE ANTES DO t-SNE
scaler_original = StandardScaler()

X_scaled_original = scaler_original.fit_transform(
    X_original
)

# t-SNE 3D
print("\nRodando t-SNE 3D...\n")

inicio_tsne = time.perf_counter()

tsne = TSNE(
    n_components=3,
    perplexity=TSNE_PERPLEXITY,
    learning_rate=TSNE_LEARNING_RATE,
    early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
    early_exaggeration=TSNE_EARLY_EXAGGERATION,
    n_iter=TSNE_N_ITER,
    exaggeration=TSNE_EXAGGERATION,
    metric=TSNE_METRIC,
    initialization=TSNE_INITIALIZATION,
    negative_gradient_method=TSNE_NEGATIVE_GRADIENT_METHOD,
    n_jobs=TSNE_N_JOBS,
    random_state=TSNE_RANDOM_STATE,
    verbose=TSNE_VERBOSE
)

X_tsne = tsne.fit(X_scaled_original)

X_tsne = np.asarray(X_tsne)

fim_tsne = time.perf_counter()

print(
    f"\nt-SNE 3D finalizado em "
    f"{(fim_tsne - inicio_tsne):.2f} segundos."
)

# NOVAS FEATURES t-SNE
df_model["TSNE_1"] = X_tsne[:, 0]
df_model["TSNE_2"] = X_tsne[:, 1]
df_model["TSNE_3"] = X_tsne[:, 2]

# FEATURES PARA O GMM
X = df_model[
    ["TSNE_1", "TSNE_2", "TSNE_3"]
]

# SCALE APÓS t-SNE
scaler_gmm = StandardScaler()

X_scaled = scaler_gmm.fit_transform(X)

# GMM
gmm = GaussianMixture(
    n_components=numero_de_componentes,
    covariance_type=tipo_matriz_covariancia,
    random_state=estado_randomico,
    reg_covar=erro_numerico,
    n_init=inicializacoes_gausianas
)

gmm.fit(X_scaled)

# CLUSTERS
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(
    clusters,
    y
)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# PROBABILIDADES
score = gmm.predict_proba(
    X_scaled
)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# PREDIÇÃO
y_pred = (
    score >= THRESHOLD
).astype(int)


# MÉTRICAS
prec, rec, _ = precision_recall_curve(
    y,
    score
)

auc_pr = auc(
    rec,
    prec
)

f1 = f1_score(
    y,
    y_pred
)

mcc = matthews_corrcoef(
    y,
    y_pred
)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(
    y,
    score
)

# MATRIZ CONFUSÃO
cm = confusion_matrix(
    y,
    y_pred
)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):
    linha = []

    for j in range(2):
        linha.append(
            f"{cm_percent[i, j]:.2f}%"
            f"<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# MATRIZ CORRELAÇÃO SPEARMAN 3x3
corr = df_model[
    ["TSNE_1", "TSNE_2", "TSNE_3"]
].corr(method="spearman")


# DATASETS SCATTER
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]


# FIGURA
fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],
        [
            {"colspan": 2},
            None
        ],
        [
            {"type": "scatter3d", "colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.25,
        0.14,
        0.61
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.08,

    subplot_titles=(
        "Correlação Spearman - t-SNE 3D",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 3D após t-SNE"
    )
)

# MATRIZ CORRELAÇÃO
fig.add_trace(
    go.Heatmap(
        z=corr.values,

        x=[
            "TSNE_1",
            "TSNE_2",
            "TSNE_3"
        ],

        y=[
            "TSNE_1",
            "TSNE_2",
            "TSNE_3"
        ],

        text=np.round(
            corr.values,
            3
        ),

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# MATRIZ CONFUSÃO
fig.add_trace(
    go.Heatmap(
        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# MÉTRICAS
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(
    go.Scatter(
        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(
            size=20
        ),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)


# SCATTER 3D NÃO FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_nao_fraude["TSNE_1"],
        y=df_nao_fraude["TSNE_2"],
        z=df_nao_fraude["TSNE_3"],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.18)",
            size=2
        )
    ),

    row=3,
    col=1
)

# SCATTER 3D FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_fraude["TSNE_1"],
        y=df_fraude["TSNE_2"],
        z=df_fraude["TSNE_3"],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=4
        )
    ),

    row=3,
    col=1
)

# AJUSTE SCENE 3D
fig.update_scenes(
    xaxis_title="TSNE_1",
    yaxis_title="TSNE_2",
    zaxis_title="TSNE_3",

    xaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    yaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    zaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    camera=dict(
        eye=dict(
            x=1.7,
            y=1.7,
            z=1.2
        )
    ),

    row=3,
    col=1
)

# LAYOUT
fig.update_layout(
    title=dict(
        text=f"""
        Relatório GMM após t-SNE 3D (Spearman)
        <br>
        Rank {RANK}
        <br>
        Features originais:
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Novas features:
        TSNE_1 vs TSNE_2 vs TSNE_3
        <br>
        Amostra: 100% fraudes + {FRAC_NAO_FRAUDE:.0%} não fraudes
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(
            size=26
        )
    ),

    width=1900,
    height=2200,

    template="plotly_white",

    font=dict(
        size=18
    ),

    margin=dict(
        t=360,
        b=160,
        l=120,
        r=120
    ),

    legend=dict(
        orientation="h",
        font=dict(size=18),
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)

# SAVE HTML
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 1
Features originais: V17 vs V11 vs V22

Quantidade usada:
status_fraude
0    28432
1      492
Name: count, dtype: int64

Rodando t-SNE 3D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=3, n_iter=1000, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 7.04 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 1.70 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.01 seconds
===> Running optimization with exaggeration=12.00, lr=2410.33 for 100 iterations...
Iteratio

### ORIGINAL

In [4]:
# CONFIG
NOME_HTML = f"3d_rank_{RANK}_orig.html"

# DIRETÓRIO
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# LOAD RANKING
df_scores = pd.read_csv("3x3_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]
feature_3 = row["Feature_3"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features: {feature_1} vs {feature_2} vs {feature_3}")

# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# DATASET AMOSTRAGEM
df_base = df[
    [feature_1, feature_2, feature_3, TARGET_COL]
].dropna()

fraudes = df_base[
    df_base[TARGET_COL] == 1
]

nao_fraudes = df_base[
    df_base[TARGET_COL] == 0
].sample(
    frac=FRAC_NAO_FRAUDE,
    random_state=estado_randomico
)

df_model = pd.concat([
    fraudes,
    nao_fraudes
])

df_model = df_model.sample(
    frac=1,
    random_state=estado_randomico
).reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# FEATURES
X = df_model[[feature_1, feature_2, feature_3]]
y = df_model[TARGET_COL]

# SCALE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# GMM
gmm = GaussianMixture(
    n_components=numero_de_componentes,
    covariance_type=tipo_matriz_covariancia,
    random_state=estado_randomico,
    reg_covar=erro_numerico,
    n_init=inicializacoes_gausianas
)

gmm.fit(X_scaled)

# CLUSTERS
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(clusters, y)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# PROBABILIDADES
score = gmm.predict_proba(X_scaled)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# PREDIÇÃO
y_pred = (score >= THRESHOLD).astype(int)

# MÉTRICAS
prec, rec, _ = precision_recall_curve(y, score)

auc_pr = auc(rec, prec)

f1 = f1_score(y, y_pred)

mcc = matthews_corrcoef(y, y_pred)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(y, score)

# MATRIZ CONFUSÃO
cm = confusion_matrix(y, y_pred)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):
    linha = []

    for j in range(2):
        linha.append(
            f"{cm_percent[i, j]:.2f}%<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# MATRIZ CORRELAÇÃO SPEARMAN 3x3
corr = df_model[
    [feature_1, feature_2, feature_3]
].corr(method="spearman")

# DATASETS SCATTER
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]

# FIGURA
fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],
        [
            {"colspan": 2},
            None
        ],
        [
            {"type": "scatter3d", "colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.26,
        0.15,
        0.59
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.09,

    subplot_titles=(
        "Correlação Spearman",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 3D das Features Originais"
    )
)

# MATRIZ CORRELAÇÃO
fig.add_trace(
    go.Heatmap(
        z=corr.values,

        x=[
            feature_1,
            feature_2,
            feature_3
        ],

        y=[
            feature_1,
            feature_2,
            feature_3
        ],

        text=np.round(corr.values, 3),

        texttemplate="%{text}",

        textfont=dict(size=18),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# MATRIZ CONFUSÃO
fig.add_trace(
    go.Heatmap(
        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(size=18),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# MÉTRICAS
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(
    go.Scatter(
        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(size=20),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

# SCATTER 3D NÃO FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_nao_fraude[feature_1],
        y=df_nao_fraude[feature_2],
        z=df_nao_fraude[feature_3],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.18)",
            size=2
        )
    ),

    row=3,
    col=1
)

# SCATTER 3D FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_fraude[feature_1],
        y=df_fraude[feature_2],
        z=df_fraude[feature_3],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=4
        )
    ),

    row=3,
    col=1
)

# AJUSTE SCENE 3D
fig.update_scenes(
    xaxis_title=feature_1,
    yaxis_title=feature_2,
    zaxis_title=feature_3,

    xaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    yaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    zaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    camera=dict(
        eye=dict(
            x=1.7,
            y=1.7,
            z=1.2
        )
    ),

    row=3,
    col=1
)

# LAYOUT
fig.update_layout(
    title=dict(
        text=f"""
        Relatório GMM 3D - Features Originais
        <br>
        Rank {RANK}
        <br>
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Amostra: 100% fraudes + {FRAC_NAO_FRAUDE:.0%} não fraudes
        <br>
        Corte: Probabilidade Cluster Fraude ≥ {THRESHOLD:.2f}
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(size=26)
    ),

    width=1900,
    height=2200,

    template="plotly_white",

    font=dict(size=18),

    margin=dict(
        t=360,
        b=160,
        l=120,
        r=120
    ),

    legend=dict(
        orientation="h",
        font=dict(size=18),
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)

# SAVE HTML
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 1
Features: V17 vs V11 vs V22

Quantidade usada:
status_fraude
0    28432
1      492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude      0    1
row_0                    
0              28098   81
1                334  411

Cluster identificado como fraude: 1

HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\3d_rank_1_orig.html


## 2 COLOCADO

### T-SNE

In [5]:
RANK = 2
NOME_HTML = f"3d_rank_{RANK}_tsne.html"
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# LOAD RANKING 3x3
df_scores = pd.read_csv("3x3_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]
feature_3 = row["Feature_3"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features originais: {feature_1} vs {feature_2} vs {feature_3}")

# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# DATASET BASE
df_base = df[
    [feature_1, feature_2, feature_3, TARGET_COL]
].dropna()


# FRAUDES: 100%
fraudes = df_base[
    df_base[TARGET_COL] == 1
]

# NÃO FRAUDES: FRAÇÃO CONFIGURÁVEL
nao_fraudes = df_base[
    df_base[TARGET_COL] == 0
].sample(
    frac=FRAC_NAO_FRAUDE,
    random_state=estado_randomico
)

# DATASET FINAL
df_model = pd.concat([
    fraudes,
    nao_fraudes
])

df_model = df_model.sample(
    frac=1,
    random_state=estado_randomico
).reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# FEATURES ORIGINAIS
X_original = df_model[
    [feature_1, feature_2, feature_3]
]

y = df_model[TARGET_COL]

# SCALE ANTES DO t-SNE
scaler_original = StandardScaler()

X_scaled_original = scaler_original.fit_transform(
    X_original
)

# t-SNE 3D
print("\nRodando t-SNE 3D...\n")

inicio_tsne = time.perf_counter()

tsne = TSNE(
    n_components=3,
    perplexity=TSNE_PERPLEXITY,
    learning_rate=TSNE_LEARNING_RATE,
    early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
    early_exaggeration=TSNE_EARLY_EXAGGERATION,
    n_iter=TSNE_N_ITER,
    exaggeration=TSNE_EXAGGERATION,
    metric=TSNE_METRIC,
    initialization=TSNE_INITIALIZATION,
    negative_gradient_method=TSNE_NEGATIVE_GRADIENT_METHOD,
    n_jobs=TSNE_N_JOBS,
    random_state=TSNE_RANDOM_STATE,
    verbose=TSNE_VERBOSE
)

X_tsne = tsne.fit(X_scaled_original)

X_tsne = np.asarray(X_tsne)

fim_tsne = time.perf_counter()

print(
    f"\nt-SNE 3D finalizado em "
    f"{(fim_tsne - inicio_tsne):.2f} segundos."
)

# NOVAS FEATURES t-SNE
df_model["TSNE_1"] = X_tsne[:, 0]
df_model["TSNE_2"] = X_tsne[:, 1]
df_model["TSNE_3"] = X_tsne[:, 2]

# FEATURES PARA O GMM
X = df_model[
    ["TSNE_1", "TSNE_2", "TSNE_3"]
]

# SCALE APÓS t-SNE
scaler_gmm = StandardScaler()

X_scaled = scaler_gmm.fit_transform(X)

# GMM
gmm = GaussianMixture(
    n_components=numero_de_componentes,
    covariance_type=tipo_matriz_covariancia,
    random_state=estado_randomico,
    reg_covar=erro_numerico,
    n_init=inicializacoes_gausianas
)

gmm.fit(X_scaled)

# CLUSTERS
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(
    clusters,
    y
)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# PROBABILIDADES
score = gmm.predict_proba(
    X_scaled
)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# PREDIÇÃO
y_pred = (
    score >= THRESHOLD
).astype(int)


# MÉTRICAS
prec, rec, _ = precision_recall_curve(
    y,
    score
)

auc_pr = auc(
    rec,
    prec
)

f1 = f1_score(
    y,
    y_pred
)

mcc = matthews_corrcoef(
    y,
    y_pred
)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(
    y,
    score
)

# MATRIZ CONFUSÃO
cm = confusion_matrix(
    y,
    y_pred
)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):
    linha = []

    for j in range(2):
        linha.append(
            f"{cm_percent[i, j]:.2f}%"
            f"<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# MATRIZ CORRELAÇÃO SPEARMAN 3x3
corr = df_model[
    ["TSNE_1", "TSNE_2", "TSNE_3"]
].corr(method="spearman")


# DATASETS SCATTER
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]


# FIGURA
fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],
        [
            {"colspan": 2},
            None
        ],
        [
            {"type": "scatter3d", "colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.25,
        0.14,
        0.61
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.08,

    subplot_titles=(
        "Correlação Spearman - t-SNE 3D",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 3D após t-SNE"
    )
)

# MATRIZ CORRELAÇÃO
fig.add_trace(
    go.Heatmap(
        z=corr.values,

        x=[
            "TSNE_1",
            "TSNE_2",
            "TSNE_3"
        ],

        y=[
            "TSNE_1",
            "TSNE_2",
            "TSNE_3"
        ],

        text=np.round(
            corr.values,
            3
        ),

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# MATRIZ CONFUSÃO
fig.add_trace(
    go.Heatmap(
        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# MÉTRICAS
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(
    go.Scatter(
        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(
            size=20
        ),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)


# SCATTER 3D NÃO FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_nao_fraude["TSNE_1"],
        y=df_nao_fraude["TSNE_2"],
        z=df_nao_fraude["TSNE_3"],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.18)",
            size=2
        )
    ),

    row=3,
    col=1
)

# SCATTER 3D FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_fraude["TSNE_1"],
        y=df_fraude["TSNE_2"],
        z=df_fraude["TSNE_3"],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=4
        )
    ),

    row=3,
    col=1
)

# AJUSTE SCENE 3D
fig.update_scenes(
    xaxis_title="TSNE_1",
    yaxis_title="TSNE_2",
    zaxis_title="TSNE_3",

    xaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    yaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    zaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    camera=dict(
        eye=dict(
            x=1.7,
            y=1.7,
            z=1.2
        )
    ),

    row=3,
    col=1
)

# LAYOUT
fig.update_layout(
    title=dict(
        text=f"""
        Relatório GMM após t-SNE 3D (Spearman)
        <br>
        Rank {RANK}
        <br>
        Features originais:
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Novas features:
        TSNE_1 vs TSNE_2 vs TSNE_3
        <br>
        Amostra: 100% fraudes + {FRAC_NAO_FRAUDE:.0%} não fraudes
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(
            size=26
        )
    ),

    width=1900,
    height=2200,

    template="plotly_white",

    font=dict(
        size=18
    ),

    margin=dict(
        t=360,
        b=160,
        l=120,
        r=120
    ),

    legend=dict(
        orientation="h",
        font=dict(size=18),
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)

# SAVE HTML
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 2
Features originais: V17 vs V11 vs V15

Quantidade usada:
status_fraude
0    28432
1      492
Name: count, dtype: int64

Rodando t-SNE 3D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=3, n_iter=1000, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 7.71 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 1.94 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.00 seconds
===> Running optimization with exaggeration=12.00, lr=2410.33 for 100 iterations...


KeyboardInterrupt: 

### ORIGINAL 

In [ ]:
# CONFIG
NOME_HTML = f"3d_rank_{RANK}_orig.html"

# DIRETÓRIO
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# LOAD RANKING
df_scores = pd.read_csv("3x3_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]
feature_3 = row["Feature_3"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features: {feature_1} vs {feature_2} vs {feature_3}")

# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# DATASET AMOSTRAGEM
df_base = df[
    [feature_1, feature_2, feature_3, TARGET_COL]
].dropna()

fraudes = df_base[
    df_base[TARGET_COL] == 1
]

nao_fraudes = df_base[
    df_base[TARGET_COL] == 0
].sample(
    frac=FRAC_NAO_FRAUDE,
    random_state=estado_randomico
)

df_model = pd.concat([
    fraudes,
    nao_fraudes
])

df_model = df_model.sample(
    frac=1,
    random_state=estado_randomico
).reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# FEATURES
X = df_model[[feature_1, feature_2, feature_3]]
y = df_model[TARGET_COL]

# SCALE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# GMM
gmm = GaussianMixture(
    n_components=numero_de_componentes,
    covariance_type=tipo_matriz_covariancia,
    random_state=estado_randomico,
    reg_covar=erro_numerico,
    n_init=inicializacoes_gausianas
)

gmm.fit(X_scaled)

# CLUSTERS
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(clusters, y)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# PROBABILIDADES
score = gmm.predict_proba(X_scaled)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# PREDIÇÃO
y_pred = (score >= THRESHOLD).astype(int)

# MÉTRICAS
prec, rec, _ = precision_recall_curve(y, score)

auc_pr = auc(rec, prec)

f1 = f1_score(y, y_pred)

mcc = matthews_corrcoef(y, y_pred)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(y, score)

# MATRIZ CONFUSÃO
cm = confusion_matrix(y, y_pred)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):
    linha = []

    for j in range(2):
        linha.append(
            f"{cm_percent[i, j]:.2f}%<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# MATRIZ CORRELAÇÃO SPEARMAN 3x3
corr = df_model[
    [feature_1, feature_2, feature_3]
].corr(method="spearman")

# DATASETS SCATTER
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]

# FIGURA
fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],
        [
            {"colspan": 2},
            None
        ],
        [
            {"type": "scatter3d", "colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.26,
        0.15,
        0.59
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.09,

    subplot_titles=(
        "Correlação Spearman",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 3D das Features Originais"
    )
)

# MATRIZ CORRELAÇÃO
fig.add_trace(
    go.Heatmap(
        z=corr.values,

        x=[
            feature_1,
            feature_2,
            feature_3
        ],

        y=[
            feature_1,
            feature_2,
            feature_3
        ],

        text=np.round(corr.values, 3),

        texttemplate="%{text}",

        textfont=dict(size=18),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# MATRIZ CONFUSÃO
fig.add_trace(
    go.Heatmap(
        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(size=18),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# MÉTRICAS
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(
    go.Scatter(
        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(size=20),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

# SCATTER 3D NÃO FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_nao_fraude[feature_1],
        y=df_nao_fraude[feature_2],
        z=df_nao_fraude[feature_3],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.18)",
            size=2
        )
    ),

    row=3,
    col=1
)

# SCATTER 3D FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_fraude[feature_1],
        y=df_fraude[feature_2],
        z=df_fraude[feature_3],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=4
        )
    ),

    row=3,
    col=1
)

# AJUSTE SCENE 3D
fig.update_scenes(
    xaxis_title=feature_1,
    yaxis_title=feature_2,
    zaxis_title=feature_3,

    xaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    yaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    zaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    camera=dict(
        eye=dict(
            x=1.7,
            y=1.7,
            z=1.2
        )
    ),

    row=3,
    col=1
)

# LAYOUT
fig.update_layout(
    title=dict(
        text=f"""
        Relatório GMM 3D - Features Originais
        <br>
        Rank {RANK}
        <br>
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Amostra: 100% fraudes + {FRAC_NAO_FRAUDE:.0%} não fraudes
        <br>
        Corte: Probabilidade Cluster Fraude ≥ {THRESHOLD:.2f}
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(size=26)
    ),

    width=1900,
    height=2200,

    template="plotly_white",

    font=dict(size=18),

    margin=dict(
        t=360,
        b=160,
        l=120,
        r=120
    ),

    legend=dict(
        orientation="h",
        font=dict(size=18),
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)

# SAVE HTML
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)

## 3 COLOCADO

### T-SNE

In [ ]:
RANK = 3
NOME_HTML = f"3d_rank_{RANK}_tsne.html"
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# LOAD RANKING 3x3
df_scores = pd.read_csv("3x3_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]
feature_3 = row["Feature_3"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features originais: {feature_1} vs {feature_2} vs {feature_3}")

# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# DATASET BASE
df_base = df[
    [feature_1, feature_2, feature_3, TARGET_COL]
].dropna()


# FRAUDES: 100%
fraudes = df_base[
    df_base[TARGET_COL] == 1
]

# NÃO FRAUDES: FRAÇÃO CONFIGURÁVEL
nao_fraudes = df_base[
    df_base[TARGET_COL] == 0
].sample(
    frac=FRAC_NAO_FRAUDE,
    random_state=estado_randomico
)

# DATASET FINAL
df_model = pd.concat([
    fraudes,
    nao_fraudes
])

df_model = df_model.sample(
    frac=1,
    random_state=estado_randomico
).reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# FEATURES ORIGINAIS
X_original = df_model[
    [feature_1, feature_2, feature_3]
]

y = df_model[TARGET_COL]

# SCALE ANTES DO t-SNE
scaler_original = StandardScaler()

X_scaled_original = scaler_original.fit_transform(
    X_original
)

# t-SNE 3D
print("\nRodando t-SNE 3D...\n")

inicio_tsne = time.perf_counter()

tsne = TSNE(
    n_components=3,
    perplexity=TSNE_PERPLEXITY,
    learning_rate=TSNE_LEARNING_RATE,
    early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
    early_exaggeration=TSNE_EARLY_EXAGGERATION,
    n_iter=TSNE_N_ITER,
    exaggeration=TSNE_EXAGGERATION,
    metric=TSNE_METRIC,
    initialization=TSNE_INITIALIZATION,
    negative_gradient_method=TSNE_NEGATIVE_GRADIENT_METHOD,
    n_jobs=TSNE_N_JOBS,
    random_state=TSNE_RANDOM_STATE,
    verbose=TSNE_VERBOSE
)

X_tsne = tsne.fit(X_scaled_original)

X_tsne = np.asarray(X_tsne)

fim_tsne = time.perf_counter()

print(
    f"\nt-SNE 3D finalizado em "
    f"{(fim_tsne - inicio_tsne):.2f} segundos."
)

# NOVAS FEATURES t-SNE
df_model["TSNE_1"] = X_tsne[:, 0]
df_model["TSNE_2"] = X_tsne[:, 1]
df_model["TSNE_3"] = X_tsne[:, 2]

# FEATURES PARA O GMM
X = df_model[
    ["TSNE_1", "TSNE_2", "TSNE_3"]
]

# SCALE APÓS t-SNE
scaler_gmm = StandardScaler()

X_scaled = scaler_gmm.fit_transform(X)

# GMM
gmm = GaussianMixture(
    n_components=numero_de_componentes,
    covariance_type=tipo_matriz_covariancia,
    random_state=estado_randomico,
    reg_covar=erro_numerico,
    n_init=inicializacoes_gausianas
)

gmm.fit(X_scaled)

# CLUSTERS
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(
    clusters,
    y
)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# PROBABILIDADES
score = gmm.predict_proba(
    X_scaled
)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# PREDIÇÃO
y_pred = (
    score >= THRESHOLD
).astype(int)


# MÉTRICAS
prec, rec, _ = precision_recall_curve(
    y,
    score
)

auc_pr = auc(
    rec,
    prec
)

f1 = f1_score(
    y,
    y_pred
)

mcc = matthews_corrcoef(
    y,
    y_pred
)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(
    y,
    score
)

# MATRIZ CONFUSÃO
cm = confusion_matrix(
    y,
    y_pred
)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):
    linha = []

    for j in range(2):
        linha.append(
            f"{cm_percent[i, j]:.2f}%"
            f"<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# MATRIZ CORRELAÇÃO SPEARMAN 3x3
corr = df_model[
    ["TSNE_1", "TSNE_2", "TSNE_3"]
].corr(method="spearman")


# DATASETS SCATTER
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]


# FIGURA
fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],
        [
            {"colspan": 2},
            None
        ],
        [
            {"type": "scatter3d", "colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.25,
        0.14,
        0.61
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.08,

    subplot_titles=(
        "Correlação Spearman - t-SNE 3D",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 3D após t-SNE"
    )
)

# MATRIZ CORRELAÇÃO
fig.add_trace(
    go.Heatmap(
        z=corr.values,

        x=[
            "TSNE_1",
            "TSNE_2",
            "TSNE_3"
        ],

        y=[
            "TSNE_1",
            "TSNE_2",
            "TSNE_3"
        ],

        text=np.round(
            corr.values,
            3
        ),

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# MATRIZ CONFUSÃO
fig.add_trace(
    go.Heatmap(
        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# MÉTRICAS
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(
    go.Scatter(
        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(
            size=20
        ),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)


# SCATTER 3D NÃO FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_nao_fraude["TSNE_1"],
        y=df_nao_fraude["TSNE_2"],
        z=df_nao_fraude["TSNE_3"],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.18)",
            size=2
        )
    ),

    row=3,
    col=1
)

# SCATTER 3D FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_fraude["TSNE_1"],
        y=df_fraude["TSNE_2"],
        z=df_fraude["TSNE_3"],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=4
        )
    ),

    row=3,
    col=1
)

# AJUSTE SCENE 3D
fig.update_scenes(
    xaxis_title="TSNE_1",
    yaxis_title="TSNE_2",
    zaxis_title="TSNE_3",

    xaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    yaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    zaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    camera=dict(
        eye=dict(
            x=1.7,
            y=1.7,
            z=1.2
        )
    ),

    row=3,
    col=1
)

# LAYOUT
fig.update_layout(
    title=dict(
        text=f"""
        Relatório GMM após t-SNE 3D (Spearman)
        <br>
        Rank {RANK}
        <br>
        Features originais:
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Novas features:
        TSNE_1 vs TSNE_2 vs TSNE_3
        <br>
        Amostra: 100% fraudes + {FRAC_NAO_FRAUDE:.0%} não fraudes
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(
            size=26
        )
    ),

    width=1900,
    height=2200,

    template="plotly_white",

    font=dict(
        size=18
    ),

    margin=dict(
        t=360,
        b=160,
        l=120,
        r=120
    ),

    legend=dict(
        orientation="h",
        font=dict(size=18),
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)

# SAVE HTML
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)

### ORIGINAL 

In [ ]:
# CONFIG
NOME_HTML = f"3d_rank_{RANK}_orig.html"

# DIRETÓRIO
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# LOAD RANKING
df_scores = pd.read_csv("3x3_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]
feature_3 = row["Feature_3"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features: {feature_1} vs {feature_2} vs {feature_3}")

# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# DATASET AMOSTRAGEM
df_base = df[
    [feature_1, feature_2, feature_3, TARGET_COL]
].dropna()

fraudes = df_base[
    df_base[TARGET_COL] == 1
]

nao_fraudes = df_base[
    df_base[TARGET_COL] == 0
].sample(
    frac=FRAC_NAO_FRAUDE,
    random_state=estado_randomico
)

df_model = pd.concat([
    fraudes,
    nao_fraudes
])

df_model = df_model.sample(
    frac=1,
    random_state=estado_randomico
).reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# FEATURES
X = df_model[[feature_1, feature_2, feature_3]]
y = df_model[TARGET_COL]

# SCALE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# GMM
gmm = GaussianMixture(
    n_components=numero_de_componentes,
    covariance_type=tipo_matriz_covariancia,
    random_state=estado_randomico,
    reg_covar=erro_numerico,
    n_init=inicializacoes_gausianas
)

gmm.fit(X_scaled)

# CLUSTERS
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(clusters, y)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# PROBABILIDADES
score = gmm.predict_proba(X_scaled)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# PREDIÇÃO
y_pred = (score >= THRESHOLD).astype(int)

# MÉTRICAS
prec, rec, _ = precision_recall_curve(y, score)

auc_pr = auc(rec, prec)

f1 = f1_score(y, y_pred)

mcc = matthews_corrcoef(y, y_pred)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(y, score)

# MATRIZ CONFUSÃO
cm = confusion_matrix(y, y_pred)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):
    linha = []

    for j in range(2):
        linha.append(
            f"{cm_percent[i, j]:.2f}%<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# MATRIZ CORRELAÇÃO SPEARMAN 3x3
corr = df_model[
    [feature_1, feature_2, feature_3]
].corr(method="spearman")

# DATASETS SCATTER
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]

# FIGURA
fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],
        [
            {"colspan": 2},
            None
        ],
        [
            {"type": "scatter3d", "colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.26,
        0.15,
        0.59
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.09,

    subplot_titles=(
        "Correlação Spearman",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 3D das Features Originais"
    )
)

# MATRIZ CORRELAÇÃO
fig.add_trace(
    go.Heatmap(
        z=corr.values,

        x=[
            feature_1,
            feature_2,
            feature_3
        ],

        y=[
            feature_1,
            feature_2,
            feature_3
        ],

        text=np.round(corr.values, 3),

        texttemplate="%{text}",

        textfont=dict(size=18),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# MATRIZ CONFUSÃO
fig.add_trace(
    go.Heatmap(
        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(size=18),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# MÉTRICAS
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(
    go.Scatter(
        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(size=20),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

# SCATTER 3D NÃO FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_nao_fraude[feature_1],
        y=df_nao_fraude[feature_2],
        z=df_nao_fraude[feature_3],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.18)",
            size=2
        )
    ),

    row=3,
    col=1
)

# SCATTER 3D FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_fraude[feature_1],
        y=df_fraude[feature_2],
        z=df_fraude[feature_3],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=4
        )
    ),

    row=3,
    col=1
)

# AJUSTE SCENE 3D
fig.update_scenes(
    xaxis_title=feature_1,
    yaxis_title=feature_2,
    zaxis_title=feature_3,

    xaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    yaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    zaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    camera=dict(
        eye=dict(
            x=1.7,
            y=1.7,
            z=1.2
        )
    ),

    row=3,
    col=1
)

# LAYOUT
fig.update_layout(
    title=dict(
        text=f"""
        Relatório GMM 3D - Features Originais
        <br>
        Rank {RANK}
        <br>
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Amostra: 100% fraudes + {FRAC_NAO_FRAUDE:.0%} não fraudes
        <br>
        Corte: Probabilidade Cluster Fraude ≥ {THRESHOLD:.2f}
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(size=26)
    ),

    width=1900,
    height=2200,

    template="plotly_white",

    font=dict(size=18),

    margin=dict(
        t=360,
        b=160,
        l=120,
        r=120
    ),

    legend=dict(
        orientation="h",
        font=dict(size=18),
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)

# SAVE HTML
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)

## 4 COLOCADO

### T-SNE

In [ ]:
RANK = 4
NOME_HTML = f"3d_rank_{RANK}_tsne.html"
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# LOAD RANKING 3x3
df_scores = pd.read_csv("3x3_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]
feature_3 = row["Feature_3"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features originais: {feature_1} vs {feature_2} vs {feature_3}")

# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# DATASET BASE
df_base = df[
    [feature_1, feature_2, feature_3, TARGET_COL]
].dropna()


# FRAUDES: 100%
fraudes = df_base[
    df_base[TARGET_COL] == 1
]

# NÃO FRAUDES: FRAÇÃO CONFIGURÁVEL
nao_fraudes = df_base[
    df_base[TARGET_COL] == 0
].sample(
    frac=FRAC_NAO_FRAUDE,
    random_state=estado_randomico
)

# DATASET FINAL
df_model = pd.concat([
    fraudes,
    nao_fraudes
])

df_model = df_model.sample(
    frac=1,
    random_state=estado_randomico
).reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# FEATURES ORIGINAIS
X_original = df_model[
    [feature_1, feature_2, feature_3]
]

y = df_model[TARGET_COL]

# SCALE ANTES DO t-SNE
scaler_original = StandardScaler()

X_scaled_original = scaler_original.fit_transform(
    X_original
)

# t-SNE 3D
print("\nRodando t-SNE 3D...\n")

inicio_tsne = time.perf_counter()

tsne = TSNE(
    n_components=3,
    perplexity=TSNE_PERPLEXITY,
    learning_rate=TSNE_LEARNING_RATE,
    early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
    early_exaggeration=TSNE_EARLY_EXAGGERATION,
    n_iter=TSNE_N_ITER,
    exaggeration=TSNE_EXAGGERATION,
    metric=TSNE_METRIC,
    initialization=TSNE_INITIALIZATION,
    negative_gradient_method=TSNE_NEGATIVE_GRADIENT_METHOD,
    n_jobs=TSNE_N_JOBS,
    random_state=TSNE_RANDOM_STATE,
    verbose=TSNE_VERBOSE
)

X_tsne = tsne.fit(X_scaled_original)

X_tsne = np.asarray(X_tsne)

fim_tsne = time.perf_counter()

print(
    f"\nt-SNE 3D finalizado em "
    f"{(fim_tsne - inicio_tsne):.2f} segundos."
)

# NOVAS FEATURES t-SNE
df_model["TSNE_1"] = X_tsne[:, 0]
df_model["TSNE_2"] = X_tsne[:, 1]
df_model["TSNE_3"] = X_tsne[:, 2]

# FEATURES PARA O GMM
X = df_model[
    ["TSNE_1", "TSNE_2", "TSNE_3"]
]

# SCALE APÓS t-SNE
scaler_gmm = StandardScaler()

X_scaled = scaler_gmm.fit_transform(X)

# GMM
gmm = GaussianMixture(
    n_components=numero_de_componentes,
    covariance_type=tipo_matriz_covariancia,
    random_state=estado_randomico,
    reg_covar=erro_numerico,
    n_init=inicializacoes_gausianas
)

gmm.fit(X_scaled)

# CLUSTERS
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(
    clusters,
    y
)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# PROBABILIDADES
score = gmm.predict_proba(
    X_scaled
)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# PREDIÇÃO
y_pred = (
    score >= THRESHOLD
).astype(int)


# MÉTRICAS
prec, rec, _ = precision_recall_curve(
    y,
    score
)

auc_pr = auc(
    rec,
    prec
)

f1 = f1_score(
    y,
    y_pred
)

mcc = matthews_corrcoef(
    y,
    y_pred
)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(
    y,
    score
)

# MATRIZ CONFUSÃO
cm = confusion_matrix(
    y,
    y_pred
)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):
    linha = []

    for j in range(2):
        linha.append(
            f"{cm_percent[i, j]:.2f}%"
            f"<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# MATRIZ CORRELAÇÃO SPEARMAN 3x3
corr = df_model[
    ["TSNE_1", "TSNE_2", "TSNE_3"]
].corr(method="spearman")


# DATASETS SCATTER
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]


# FIGURA
fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],
        [
            {"colspan": 2},
            None
        ],
        [
            {"type": "scatter3d", "colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.25,
        0.14,
        0.61
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.08,

    subplot_titles=(
        "Correlação Spearman - t-SNE 3D",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 3D após t-SNE"
    )
)

# MATRIZ CORRELAÇÃO
fig.add_trace(
    go.Heatmap(
        z=corr.values,

        x=[
            "TSNE_1",
            "TSNE_2",
            "TSNE_3"
        ],

        y=[
            "TSNE_1",
            "TSNE_2",
            "TSNE_3"
        ],

        text=np.round(
            corr.values,
            3
        ),

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# MATRIZ CONFUSÃO
fig.add_trace(
    go.Heatmap(
        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# MÉTRICAS
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(
    go.Scatter(
        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(
            size=20
        ),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)


# SCATTER 3D NÃO FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_nao_fraude["TSNE_1"],
        y=df_nao_fraude["TSNE_2"],
        z=df_nao_fraude["TSNE_3"],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.18)",
            size=2
        )
    ),

    row=3,
    col=1
)

# SCATTER 3D FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_fraude["TSNE_1"],
        y=df_fraude["TSNE_2"],
        z=df_fraude["TSNE_3"],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=4
        )
    ),

    row=3,
    col=1
)

# AJUSTE SCENE 3D
fig.update_scenes(
    xaxis_title="TSNE_1",
    yaxis_title="TSNE_2",
    zaxis_title="TSNE_3",

    xaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    yaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    zaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    camera=dict(
        eye=dict(
            x=1.7,
            y=1.7,
            z=1.2
        )
    ),

    row=3,
    col=1
)

# LAYOUT
fig.update_layout(
    title=dict(
        text=f"""
        Relatório GMM após t-SNE 3D (Spearman)
        <br>
        Rank {RANK}
        <br>
        Features originais:
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Novas features:
        TSNE_1 vs TSNE_2 vs TSNE_3
        <br>
        Amostra: 100% fraudes + {FRAC_NAO_FRAUDE:.0%} não fraudes
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(
            size=26
        )
    ),

    width=1900,
    height=2200,

    template="plotly_white",

    font=dict(
        size=18
    ),

    margin=dict(
        t=360,
        b=160,
        l=120,
        r=120
    ),

    legend=dict(
        orientation="h",
        font=dict(size=18),
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)

# SAVE HTML
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)

### ORIGINAL 

In [ ]:
# CONFIG
NOME_HTML = f"3d_rank_{RANK}_orig.html"

# DIRETÓRIO
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# LOAD RANKING
df_scores = pd.read_csv("3x3_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]
feature_3 = row["Feature_3"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features: {feature_1} vs {feature_2} vs {feature_3}")

# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# DATASET AMOSTRAGEM
df_base = df[
    [feature_1, feature_2, feature_3, TARGET_COL]
].dropna()

fraudes = df_base[
    df_base[TARGET_COL] == 1
]

nao_fraudes = df_base[
    df_base[TARGET_COL] == 0
].sample(
    frac=FRAC_NAO_FRAUDE,
    random_state=estado_randomico
)

df_model = pd.concat([
    fraudes,
    nao_fraudes
])

df_model = df_model.sample(
    frac=1,
    random_state=estado_randomico
).reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# FEATURES
X = df_model[[feature_1, feature_2, feature_3]]
y = df_model[TARGET_COL]

# SCALE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# GMM
gmm = GaussianMixture(
    n_components=numero_de_componentes,
    covariance_type=tipo_matriz_covariancia,
    random_state=estado_randomico,
    reg_covar=erro_numerico,
    n_init=inicializacoes_gausianas
)

gmm.fit(X_scaled)

# CLUSTERS
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(clusters, y)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# PROBABILIDADES
score = gmm.predict_proba(X_scaled)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# PREDIÇÃO
y_pred = (score >= THRESHOLD).astype(int)

# MÉTRICAS
prec, rec, _ = precision_recall_curve(y, score)

auc_pr = auc(rec, prec)

f1 = f1_score(y, y_pred)

mcc = matthews_corrcoef(y, y_pred)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(y, score)

# MATRIZ CONFUSÃO
cm = confusion_matrix(y, y_pred)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):
    linha = []

    for j in range(2):
        linha.append(
            f"{cm_percent[i, j]:.2f}%<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# MATRIZ CORRELAÇÃO SPEARMAN 3x3
corr = df_model[
    [feature_1, feature_2, feature_3]
].corr(method="spearman")

# DATASETS SCATTER
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]

# FIGURA
fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],
        [
            {"colspan": 2},
            None
        ],
        [
            {"type": "scatter3d", "colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.26,
        0.15,
        0.59
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.09,

    subplot_titles=(
        "Correlação Spearman",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 3D das Features Originais"
    )
)

# MATRIZ CORRELAÇÃO
fig.add_trace(
    go.Heatmap(
        z=corr.values,

        x=[
            feature_1,
            feature_2,
            feature_3
        ],

        y=[
            feature_1,
            feature_2,
            feature_3
        ],

        text=np.round(corr.values, 3),

        texttemplate="%{text}",

        textfont=dict(size=18),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# MATRIZ CONFUSÃO
fig.add_trace(
    go.Heatmap(
        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(size=18),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# MÉTRICAS
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(
    go.Scatter(
        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(size=20),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

# SCATTER 3D NÃO FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_nao_fraude[feature_1],
        y=df_nao_fraude[feature_2],
        z=df_nao_fraude[feature_3],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.18)",
            size=2
        )
    ),

    row=3,
    col=1
)

# SCATTER 3D FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_fraude[feature_1],
        y=df_fraude[feature_2],
        z=df_fraude[feature_3],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=4
        )
    ),

    row=3,
    col=1
)

# AJUSTE SCENE 3D
fig.update_scenes(
    xaxis_title=feature_1,
    yaxis_title=feature_2,
    zaxis_title=feature_3,

    xaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    yaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    zaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    camera=dict(
        eye=dict(
            x=1.7,
            y=1.7,
            z=1.2
        )
    ),

    row=3,
    col=1
)

# LAYOUT
fig.update_layout(
    title=dict(
        text=f"""
        Relatório GMM 3D - Features Originais
        <br>
        Rank {RANK}
        <br>
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Amostra: 100% fraudes + {FRAC_NAO_FRAUDE:.0%} não fraudes
        <br>
        Corte: Probabilidade Cluster Fraude ≥ {THRESHOLD:.2f}
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(size=26)
    ),

    width=1900,
    height=2200,

    template="plotly_white",

    font=dict(size=18),

    margin=dict(
        t=360,
        b=160,
        l=120,
        r=120
    ),

    legend=dict(
        orientation="h",
        font=dict(size=18),
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)

# SAVE HTML
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)

## 5 COLOCADO

### T-SNE

In [ ]:
RANK = 5
NOME_HTML = f"3d_rank_{RANK}_tsne.html"
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# LOAD RANKING 3x3
df_scores = pd.read_csv("3x3_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]
feature_3 = row["Feature_3"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features originais: {feature_1} vs {feature_2} vs {feature_3}")

# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# DATASET BASE
df_base = df[
    [feature_1, feature_2, feature_3, TARGET_COL]
].dropna()


# FRAUDES: 100%
fraudes = df_base[
    df_base[TARGET_COL] == 1
]

# NÃO FRAUDES: FRAÇÃO CONFIGURÁVEL
nao_fraudes = df_base[
    df_base[TARGET_COL] == 0
].sample(
    frac=FRAC_NAO_FRAUDE,
    random_state=estado_randomico
)

# DATASET FINAL
df_model = pd.concat([
    fraudes,
    nao_fraudes
])

df_model = df_model.sample(
    frac=1,
    random_state=estado_randomico
).reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# FEATURES ORIGINAIS
X_original = df_model[
    [feature_1, feature_2, feature_3]
]

y = df_model[TARGET_COL]

# SCALE ANTES DO t-SNE
scaler_original = StandardScaler()

X_scaled_original = scaler_original.fit_transform(
    X_original
)

# t-SNE 3D
print("\nRodando t-SNE 3D...\n")

inicio_tsne = time.perf_counter()

tsne = TSNE(
    n_components=3,
    perplexity=TSNE_PERPLEXITY,
    learning_rate=TSNE_LEARNING_RATE,
    early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
    early_exaggeration=TSNE_EARLY_EXAGGERATION,
    n_iter=TSNE_N_ITER,
    exaggeration=TSNE_EXAGGERATION,
    metric=TSNE_METRIC,
    initialization=TSNE_INITIALIZATION,
    negative_gradient_method=TSNE_NEGATIVE_GRADIENT_METHOD,
    n_jobs=TSNE_N_JOBS,
    random_state=TSNE_RANDOM_STATE,
    verbose=TSNE_VERBOSE
)

X_tsne = tsne.fit(X_scaled_original)

X_tsne = np.asarray(X_tsne)

fim_tsne = time.perf_counter()

print(
    f"\nt-SNE 3D finalizado em "
    f"{(fim_tsne - inicio_tsne):.2f} segundos."
)

# NOVAS FEATURES t-SNE
df_model["TSNE_1"] = X_tsne[:, 0]
df_model["TSNE_2"] = X_tsne[:, 1]
df_model["TSNE_3"] = X_tsne[:, 2]

# FEATURES PARA O GMM
X = df_model[
    ["TSNE_1", "TSNE_2", "TSNE_3"]
]

# SCALE APÓS t-SNE
scaler_gmm = StandardScaler()

X_scaled = scaler_gmm.fit_transform(X)

# GMM
gmm = GaussianMixture(
    n_components=numero_de_componentes,
    covariance_type=tipo_matriz_covariancia,
    random_state=estado_randomico,
    reg_covar=erro_numerico,
    n_init=inicializacoes_gausianas
)

gmm.fit(X_scaled)

# CLUSTERS
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(
    clusters,
    y
)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# PROBABILIDADES
score = gmm.predict_proba(
    X_scaled
)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# PREDIÇÃO
y_pred = (
    score >= THRESHOLD
).astype(int)


# MÉTRICAS
prec, rec, _ = precision_recall_curve(
    y,
    score
)

auc_pr = auc(
    rec,
    prec
)

f1 = f1_score(
    y,
    y_pred
)

mcc = matthews_corrcoef(
    y,
    y_pred
)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(
    y,
    score
)

# MATRIZ CONFUSÃO
cm = confusion_matrix(
    y,
    y_pred
)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):
    linha = []

    for j in range(2):
        linha.append(
            f"{cm_percent[i, j]:.2f}%"
            f"<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# MATRIZ CORRELAÇÃO SPEARMAN 3x3
corr = df_model[
    ["TSNE_1", "TSNE_2", "TSNE_3"]
].corr(method="spearman")


# DATASETS SCATTER
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]


# FIGURA
fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],
        [
            {"colspan": 2},
            None
        ],
        [
            {"type": "scatter3d", "colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.25,
        0.14,
        0.61
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.08,

    subplot_titles=(
        "Correlação Spearman - t-SNE 3D",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 3D após t-SNE"
    )
)

# MATRIZ CORRELAÇÃO
fig.add_trace(
    go.Heatmap(
        z=corr.values,

        x=[
            "TSNE_1",
            "TSNE_2",
            "TSNE_3"
        ],

        y=[
            "TSNE_1",
            "TSNE_2",
            "TSNE_3"
        ],

        text=np.round(
            corr.values,
            3
        ),

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# MATRIZ CONFUSÃO
fig.add_trace(
    go.Heatmap(
        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# MÉTRICAS
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(
    go.Scatter(
        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(
            size=20
        ),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)


# SCATTER 3D NÃO FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_nao_fraude["TSNE_1"],
        y=df_nao_fraude["TSNE_2"],
        z=df_nao_fraude["TSNE_3"],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.18)",
            size=2
        )
    ),

    row=3,
    col=1
)

# SCATTER 3D FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_fraude["TSNE_1"],
        y=df_fraude["TSNE_2"],
        z=df_fraude["TSNE_3"],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=4
        )
    ),

    row=3,
    col=1
)

# AJUSTE SCENE 3D
fig.update_scenes(
    xaxis_title="TSNE_1",
    yaxis_title="TSNE_2",
    zaxis_title="TSNE_3",

    xaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    yaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    zaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    camera=dict(
        eye=dict(
            x=1.7,
            y=1.7,
            z=1.2
        )
    ),

    row=3,
    col=1
)

# LAYOUT
fig.update_layout(
    title=dict(
        text=f"""
        Relatório GMM após t-SNE 3D (Spearman)
        <br>
        Rank {RANK}
        <br>
        Features originais:
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Novas features:
        TSNE_1 vs TSNE_2 vs TSNE_3
        <br>
        Amostra: 100% fraudes + {FRAC_NAO_FRAUDE:.0%} não fraudes
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(
            size=26
        )
    ),

    width=1900,
    height=2200,

    template="plotly_white",

    font=dict(
        size=18
    ),

    margin=dict(
        t=360,
        b=160,
        l=120,
        r=120
    ),

    legend=dict(
        orientation="h",
        font=dict(size=18),
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)

# SAVE HTML
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)

### ORIGINAL 

In [ ]:
# CONFIG
NOME_HTML = f"3d_rank_{RANK}_orig.html"

# DIRETÓRIO
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# LOAD RANKING
df_scores = pd.read_csv("3x3_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]
feature_3 = row["Feature_3"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features: {feature_1} vs {feature_2} vs {feature_3}")

# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# DATASET AMOSTRAGEM
df_base = df[
    [feature_1, feature_2, feature_3, TARGET_COL]
].dropna()

fraudes = df_base[
    df_base[TARGET_COL] == 1
]

nao_fraudes = df_base[
    df_base[TARGET_COL] == 0
].sample(
    frac=FRAC_NAO_FRAUDE,
    random_state=estado_randomico
)

df_model = pd.concat([
    fraudes,
    nao_fraudes
])

df_model = df_model.sample(
    frac=1,
    random_state=estado_randomico
).reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# FEATURES
X = df_model[[feature_1, feature_2, feature_3]]
y = df_model[TARGET_COL]

# SCALE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# GMM
gmm = GaussianMixture(
    n_components=numero_de_componentes,
    covariance_type=tipo_matriz_covariancia,
    random_state=estado_randomico,
    reg_covar=erro_numerico,
    n_init=inicializacoes_gausianas
)

gmm.fit(X_scaled)

# CLUSTERS
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(clusters, y)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# PROBABILIDADES
score = gmm.predict_proba(X_scaled)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# PREDIÇÃO
y_pred = (score >= THRESHOLD).astype(int)

# MÉTRICAS
prec, rec, _ = precision_recall_curve(y, score)

auc_pr = auc(rec, prec)

f1 = f1_score(y, y_pred)

mcc = matthews_corrcoef(y, y_pred)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(y, score)

# MATRIZ CONFUSÃO
cm = confusion_matrix(y, y_pred)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):
    linha = []

    for j in range(2):
        linha.append(
            f"{cm_percent[i, j]:.2f}%<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# MATRIZ CORRELAÇÃO SPEARMAN 3x3
corr = df_model[
    [feature_1, feature_2, feature_3]
].corr(method="spearman")

# DATASETS SCATTER
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]

# FIGURA
fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],
        [
            {"colspan": 2},
            None
        ],
        [
            {"type": "scatter3d", "colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.26,
        0.15,
        0.59
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.09,

    subplot_titles=(
        "Correlação Spearman",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 3D das Features Originais"
    )
)

# MATRIZ CORRELAÇÃO
fig.add_trace(
    go.Heatmap(
        z=corr.values,

        x=[
            feature_1,
            feature_2,
            feature_3
        ],

        y=[
            feature_1,
            feature_2,
            feature_3
        ],

        text=np.round(corr.values, 3),

        texttemplate="%{text}",

        textfont=dict(size=18),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# MATRIZ CONFUSÃO
fig.add_trace(
    go.Heatmap(
        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(size=18),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# MÉTRICAS
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(
    go.Scatter(
        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(size=20),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

# SCATTER 3D NÃO FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_nao_fraude[feature_1],
        y=df_nao_fraude[feature_2],
        z=df_nao_fraude[feature_3],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.18)",
            size=2
        )
    ),

    row=3,
    col=1
)

# SCATTER 3D FRAUDE
fig.add_trace(
    go.Scatter3d(
        x=df_fraude[feature_1],
        y=df_fraude[feature_2],
        z=df_fraude[feature_3],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=4
        )
    ),

    row=3,
    col=1
)

# AJUSTE SCENE 3D
fig.update_scenes(
    xaxis_title=feature_1,
    yaxis_title=feature_2,
    zaxis_title=feature_3,

    xaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    yaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    zaxis=dict(
        backgroundcolor="white",
        gridcolor="lightgray",
        zerolinecolor="lightgray"
    ),

    camera=dict(
        eye=dict(
            x=1.7,
            y=1.7,
            z=1.2
        )
    ),

    row=3,
    col=1
)

# LAYOUT
fig.update_layout(
    title=dict(
        text=f"""
        Relatório GMM 3D - Features Originais
        <br>
        Rank {RANK}
        <br>
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Amostra: 100% fraudes + {FRAC_NAO_FRAUDE:.0%} não fraudes
        <br>
        Corte: Probabilidade Cluster Fraude ≥ {THRESHOLD:.2f}
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(size=26)
    ),

    width=1900,
    height=2200,

    template="plotly_white",

    font=dict(size=18),

    margin=dict(
        t=360,
        b=160,
        l=120,
        r=120
    ),

    legend=dict(
        orientation="h",
        font=dict(size=18),
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)

# SAVE HTML
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)